# 00 - data audit, `aaer`

Coverage, row counts, dtypes, units, missingness, duplicates and the mapping losses for the
three things this project rests on: the SEC Financial Statement Data Sets quarters, the Bao
et al. (2020) replication files, and the AAER release listing.

**There is no inference in this notebook and there must never be any.** Nothing here fits a
model, tests a hypothesis or computes a detection statistic. No Beneish index is computed,
no M-score, no classifier and no metric. Its whole job is to establish what is on disk, how
much of it survives each mapping step, and where the holes are, so that the tests in
`src/aaer/analysis/` can later be run against something known. The replication targets are
in `docs/validation_anchors.md`; the confounds are in `docs/known_traps.md`; every claim
about a source is in `data/SOURCES.yaml`.

If `data/raw` is empty, every cell that reads data prints what to run and does nothing else.
The catalogue cells (the tag map, the twelve required items) describe code rather than data
and run either way.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import aaer
import pandas as pd
from aaer.acquire.fsds import (
    FSDS_FIRST_QUARTER,
    FSDS_FIRST_QUARTER_WITH_ROWS,
    FSDS_LATEST_VERIFIED_QUARTER,
    quarters,
)
from aaer.clean import add_lags, load_listing, map_beneish_items, mapping_table, read_fsds_zip
from aaer.clean.xbrl_map import (
    BENEISH_TAG_MAP,
    CONFIRMED_ITEMS,
    UNCONFIRMED_ITEMS,
    check_mapping_covers_beneish,
)
from aaer.features.beneish import LAGGED_COLUMNS, REQUIRED_COLUMNS
from forensics_core.provenance import load_sources

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)

PROJECT = Path(aaer.__file__).resolve().parents[2]
DATA = PROJECT / "data"
RAW = DATA / "raw"

RUN_FIRST = (
    "data/raw is empty. From projects/aaer run:\n"
    "    make data     # acquire the sources listed in data/SOURCES.yaml\n"
    "and then re-run this notebook. Nothing is computed until then."
)

RAW_FILES = (
    sorted(p for p in RAW.rglob("*") if p.is_file() and p.name != ".gitkeep")
    if RAW.is_dir()
    else []
)
RAW_EMPTY = not RAW_FILES
if RAW_EMPTY:
    print(RUN_FIRST)

## What is on disk, and what the registry says should be

Two separate questions. The first is a directory listing. The second is what
`data/SOURCES.yaml` records: 42 entries, each carrying the access tier, the reachability
status a real fetch attempt established, and a `local_path` that stays null until the file
actually lands. An entry that names a `local_path` no longer on disk is a broken acquisition,
not a missing source, and the two are separated below.

In [ ]:
if RAW_EMPTY:
    print(RUN_FIRST)
else:
    inventory = pd.DataFrame(
        [
            {
                "directory": p.parent.relative_to(RAW).as_posix() or ".",
                "suffix": p.suffix.lower(),
                "bytes": p.stat().st_size,
            }
            for p in RAW_FILES
        ]
    )
    display(
        inventory.groupby("directory")
        .agg(n_files=("bytes", "size"), total_bytes=("bytes", "sum"))
        .sort_index()
    )
    print(f"{len(RAW_FILES):,} files, {int(inventory['bytes'].sum()):,} bytes under data/raw")

In [ ]:
SOURCES = load_sources(DATA / "SOURCES.yaml")
registry = pd.DataFrame(
    [
        {
            "id": s.id,
            "access": s.access,
            "status": s.status,
            "local_path": s.local_path or "",
            "recorded_bytes": s.bytes,
            "on_disk": bool(s.local_path) and (PROJECT / s.local_path).exists(),
        }
        for s in SOURCES
    ]
)
print(f"{len(registry)} registry entries in data/SOURCES.yaml")
display(pd.crosstab(registry["status"], registry["access"], margins=True))
acquired = registry.loc[registry["local_path"].ne("")]
print(f"{len(acquired)} entries record a local_path; {int(acquired['on_disk'].sum())} are on disk")
if len(acquired) and not acquired["on_disk"].all():
    print("recorded but absent:")
    display(acquired.loc[~acquired["on_disk"], ["id", "local_path", "recorded_bytes"]])

## Which quarters of the Financial Statement Data Sets are present

The expected range is fixed in `aaer.acquire.fsds`: `2009q1`, which the SEC documentation
describes as column headings with no rows, followed by every quarter from `2009q2` to the last
one the registry records as confirmed to exist. Nothing before 15 April 2009 exists in this
series at any price, which is the constraint `data/ACCESS_NOTES.md` opens with.

The next cell makes one pass over every quarterly zip on disk. It reports the four table row
counts, any header that differs from the documented one, and, for the same quarter, what the
tag mapping resolved. One pass rather than two: recent quarters are around 120 MB each and
reading the whole series twice would double the wait for nothing.

In [ ]:
EXPECTED_QUARTERS = [FSDS_FIRST_QUARTER, *quarters()]
FSDS_DIR = RAW / "fsds"
FSDS_ZIPS = sorted(FSDS_DIR.glob("*.zip"), key=lambda p: p.stem) if FSDS_DIR.is_dir() else []
on_disk = {p.stem.lower() for p in FSDS_ZIPS}

if RAW_EMPTY:
    print(RUN_FIRST)
else:
    missing = [q for q in EXPECTED_QUARTERS if q not in on_disk]
    unexpected = sorted(on_disk - set(EXPECTED_QUARTERS))
    print(
        f"expected {len(EXPECTED_QUARTERS)} quarters, "
        f"{EXPECTED_QUARTERS[0]} to {FSDS_LATEST_VERIFIED_QUARTER}"
    )
    print(
        f"on disk {len(on_disk)}, missing {len(missing)}, not in the expected range "
        f"{len(unexpected)}"
    )
    print("missing:", missing)
    print("unexpected:", unexpected)
    print(
        f"{FSDS_FIRST_QUARTER} is the documented empty placeholder; "
        f"{FSDS_FIRST_QUARTER_WITH_ROWS} is the first quarter with rows"
    )

In [ ]:
QUARTER_ROWS = None
MAP_STEPS = None
ITEM_COVERAGE = None
PANEL = None

if not FSDS_ZIPS:
    print(RUN_FIRST if RAW_EMPTY else "no quarterly zips under data/raw/fsds")
else:
    counts, steps, coverage, panels = [], [], [], []
    for path in FSDS_ZIPS:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            qtr = read_fsds_zip(path)
        row = {"quarter": qtr.quarter, **qtr.row_counts()}
        row["is_empty"] = qtr.is_empty
        row["header_mismatches"] = ";".join(sorted(qtr.header_mismatches))
        row["warnings"] = len(caught)
        row["zip_bytes"] = path.stat().st_size
        counts.append(row)
        if qtr.is_empty:
            continue
        result = map_beneish_items(qtr.sub, qtr.num)
        steps.append(result.steps.assign(quarter=qtr.quarter))
        coverage.append(result.item_coverage.assign(quarter=qtr.quarter))
        panels.append(result.frame.assign(quarter=qtr.quarter))
        if result.unmapped_items:
            print(
                f"{qtr.quarter}: no fact matched any candidate tag for "
                f"{list(result.unmapped_items)}"
            )
    QUARTER_ROWS = pd.DataFrame(counts)
    MAP_STEPS = pd.concat(steps, ignore_index=True) if steps else None
    ITEM_COVERAGE = pd.concat(coverage, ignore_index=True) if coverage else None
    PANEL = pd.concat(panels, ignore_index=True) if panels else None
    display(QUARTER_ROWS)
    print(
        f"quarters with a header differing from the documented one: "
        f"{int((QUARTER_ROWS['header_mismatches'] != '').sum())}"
    )

## The twelve items, and how much of the tag map is guesswork

`aaer.features.beneish.REQUIRED_COLUMNS` names twelve financial items for year `t`, ten of
which are also needed for `t-1`. `aaer.clean.xbrl_map.BENEISH_TAG_MAP` maps each to us-gaap
element names in priority order.

**Eleven of the twelve mappings are unconfirmed.** Exactly one element name in the table was
observed in a response actually fetched from the SEC; the rest are the implementer's knowledge
of the taxonomy, written down and flagged. Three selection rules are unconfirmed too: what
`qtrs` means, the `ddate == period` convention, and the exclusion of filer extension tags by
`version` prefix. The count below is a property of the code, not of any data, so this cell
runs whether or not anything has been downloaded.

In [ ]:
print(f"{len(REQUIRED_COLUMNS)} required items; {len(LAGGED_COLUMNS)} of them also needed for t-1")
print(f"{len(BENEISH_TAG_MAP)} items in BENEISH_TAG_MAP")
print(f"confirmed against a fetched SEC response: {len(CONFIRMED_ITEMS)} {list(CONFIRMED_ITEMS)}")
print(f"assumed and never checked against a filing: {len(UNCONFIRMED_ITEMS)}")
print("required items with no entry in the map:", list(check_mapping_covers_beneish()))
print("items needed for t only:", [c for c in REQUIRED_COLUMNS if f"{c}_lag" not in LAGGED_COLUMNS])
display(mapping_table().drop(columns=["note"]))

## How many firm-years are lost at each mapping step

`map_beneish_items` returns the loss accounting rather than only its output: submissions in,
annual filings deduplicated to one per `(cik, fy)`, firm-years with at least one mapped fact,
firm-years complete on all twelve items. The counts below are summed over the quarters on
disk. They are counts, not a judgement: an item that maps to nothing means the mapping is
wrong, not that the item does not exist.

`tags_used` is the diagnostic the data dictionary asks for. Reading it for an early quarter
and a late one is how the ASC 606 revenue-tag change becomes visible, which is why both ends
of the range are printed rather than a pooled string.

In [ ]:
if MAP_STEPS is None or PANEL is None:
    print(RUN_FIRST if RAW_EMPTY else "no non-empty quarter was read, so nothing was mapped")
else:
    pooled = MAP_STEPS.groupby("step", sort=False)[["n_rows", "n_lost"]].sum()
    display(pooled)
    per_item = (
        ITEM_COVERAGE.groupby("item", sort=False)[["n_present", "n_missing"]]
        .sum()
        .reindex(list(REQUIRED_COLUMNS))
    )
    per_item["share_present"] = per_item["n_present"] / (
        per_item["n_present"] + per_item["n_missing"]
    )
    per_item["confirmed"] = [BENEISH_TAG_MAP[i].confirmed for i in per_item.index]
    display(per_item)
    print("items with no value in any quarter:", list(per_item.index[per_item["n_present"] == 0]))
    edges = [ITEM_COVERAGE["quarter"].min(), ITEM_COVERAGE["quarter"].max()]
    tags = ITEM_COVERAGE.loc[
        ITEM_COVERAGE["quarter"].isin(edges) & ITEM_COVERAGE["tags_used"].ne(""),
        ["quarter", "item", "tags_used"],
    ]
    display(tags.sort_values(["quarter", "item"]))
    duplicated = PANEL.duplicated(subset=["cik", "fy"], keep=False)
    print(
        f"firm-year rows: {len(PANEL):,}; rows sharing a (cik, fy) across quarters: "
        f"{int(duplicated.sum()):,}"
    )
    display(PANEL[list(REQUIRED_COLUMNS)].dtypes.rename("dtype").to_frame())

In [ ]:
if PANEL is None or PANEL.empty:
    print(RUN_FIRST if RAW_EMPTY else "no firm-years were mapped, so there is nothing to lag")
else:
    lagged = add_lags(PANEL)
    print(lagged.summary())
    frame = lagged.frame
    frame = frame.assign(
        has_all_lags=frame[list(LAGGED_COLUMNS)].notna().all(axis=1),
        complete_t=frame[list(REQUIRED_COLUMNS)].notna().all(axis=1),
    )
    display(
        frame.groupby("fy").agg(
            firm_years=("cik", "size"),
            complete_on_year_t=("complete_t", "sum"),
            with_all_lags=("has_all_lags", "sum"),
        )
    )

## What the Bao et al. replication file cannot support, and why

`data_FraudDetection_JAR2020.csv` ships 28 raw Compustat annual items for 146,045 firm-years
over fiscal 1990-2014 with the labels attached, so the published benchmark can be reproduced
without a Compustat subscription. The Beneish M-score is a different matter. Three of the
twelve inputs are absent, and `docs/data_dictionary.md` records which:

| Beneish component | From this file? | Blocked by |
|---|---|---|
| DSRI, GMI, SGI, LVGI | yes | |
| TATA | only with the balance-sheet definition Beneish (1999) actually uses | `oancf` absent |
| AQI, DEPI | no | `ppent` absent; only gross `ppegt` is shipped, and `xbrl_map` refuses a gross-for-net substitution |
| SGAI | no | `xsga` absent |

So four of the eight components as shipped, five with a balance-sheet TATA. The file also
stops at fiscal 2014. The item-to-Compustat column names below are copied from section 1 of
`docs/data_dictionary.md`; the cell checks which of them the file on disk actually carries
rather than assuming the documented answer.

In [ ]:
BAO_DIR = RAW / "bao2020"
BAO_ANALYSIS = BAO_DIR / "data_FraudDetection_JAR2020.csv"
BAO_LABELS = BAO_DIR / "AAER_firm_year.csv"

# From docs/data_dictionary.md section 1. Context for a replication argument, not a mapping
# this project performs: the free paths are the SEC data sets and this file.
BENEISH_TO_COMPUSTAT = {
    "receivables": "rect",
    "sales": "sale",
    "cogs": "cogs",
    "current_assets": "act",
    "ppe_net": "ppent",
    "total_assets": "at",
    "depreciation": "dp",
    "sga": "xsga",
    "long_term_debt": "dltt",
    "current_liabilities": "lct",
    "income_continuing_ops": "ib",
    "cash_from_operations": "oancf",
}

BAO = None
if not BAO_ANALYSIS.is_file():
    print(RUN_FIRST if RAW_EMPTY else f"not on disk: {BAO_ANALYSIS.relative_to(PROJECT)}")
else:
    BAO = pd.read_csv(BAO_ANALYSIS)
    print(f"{len(BAO):,} rows, {BAO.shape[1]} columns, {BAO_ANALYSIS.stat().st_size:,} bytes")
    items = pd.DataFrame(
        [
            {"beneish_item": item, "compustat_column": col, "in_file": col in BAO.columns}
            for item, col in BENEISH_TO_COMPUSTAT.items()
        ]
    )
    display(items)
    print(
        "Beneish inputs absent from this file:",
        list(items.loc[~items["in_file"], "compustat_column"]),
    )
    if "fyear" in BAO.columns:
        by_year = BAO.groupby("fyear").agg(firm_years=("fyear", "size"))
        if "gvkey" in BAO.columns:
            by_year["firms"] = BAO.groupby("fyear")["gvkey"].nunique()
        display(by_year)
        print(f"fiscal years {int(BAO['fyear'].min())} to {int(BAO['fyear'].max())}")
    key = [c for c in ("gvkey", "fyear") if c in BAO.columns]
    if len(key) == 2:
        print(f"rows duplicating (gvkey, fyear): {int(BAO.duplicated(subset=key).sum()):,}")

## The AAER release listing, and its coverage by year

One row per enforcement release: AAER number, date, respondent, the Securities Act and
Exchange Act release numbers, and the order PDF. Two cautions travel with it. The parser was
developed against a synthetic fixture and has not been run against a real page, so
`n_dropped` and `n_rows_seen` are the first things to read. And the date is the **release**
date, not the date of the misstatement, so the year grid below is a publication schedule, not
a distribution of violations.

The listing gives a respondent name and no CIK, so joining this table to the financial side is
an open task rather than a solved one. Nothing is joined here.

In [ ]:
LISTING_DIR = RAW / "aaer_listing"
listing_pages = sorted(LISTING_DIR.glob("page_*.html")) if LISTING_DIR.is_dir() else []
if not listing_pages:
    print(RUN_FIRST if RAW_EMPTY else "no saved pages under data/raw/aaer_listing")
else:
    parse = load_listing(LISTING_DIR)
    print(f"{len(listing_pages)} saved pages")
    print(parse.summary())
    print("items the page itself reports:", parse.total_items)
    releases = parse.releases
    display(releases.dtypes.rename("dtype").to_frame())
    by_year = (
        releases.assign(release_year=releases["date"].dt.year)
        .groupby("release_year", dropna=False)
        .agg(releases=("respondent", "size"), with_aaer_number=("aaer_number", "count"))
    )
    display(by_year)
    print(f"releases with no parsed date: {int(releases['date'].isna().sum()):,}")
    print(f"releases with no PDF link: {int(releases['pdf_url'].isna().sum()):,}")
    print(f"repeated AAER numbers: {int(releases['aaer_number'].dropna().duplicated().sum()):,}")

## The base rate, as a raw count and a proportion

Two numbers and nothing else. They are here because every table this project ever produces has
to print them beside itself.

**Accuracy is meaningless at this base rate.** A model that flags nothing is over 99 per cent
accurate. That is why `docs/known_traps.md` trap 2 permits only rank metrics, and why the
published benchmark reports NDCG at k equal to 1 per cent of test firm-years.

**And the zeros are not zeros.** An unflagged firm-year is one nobody charged, not one that was
clean. Treating the unflagged rows as negatives would teach a classifier that every undetected
misstatement is an example of honesty; the positive-unlabeled formulation exists for exactly
that reason. The proportion below is the share of firm-years the SEC prosecuted, which is not
the share that misstated.

In [ ]:
if BAO is None:
    print(RUN_FIRST if RAW_EMPTY else "the Bao et al. analysis file is not on disk")
elif "misstate" not in BAO.columns:
    print("the analysis file on disk carries no 'misstate' column; nothing to count")
else:
    n_rows = len(BAO)
    n_pos = int(BAO["misstate"].sum())
    print(f"labelled positives: {n_pos:,} of {n_rows:,} firm-years ({n_pos / n_rows:.4%})")
    print(f"unflagged firm-years: {n_rows - n_pos:,}  (unlabeled, not negative)")
    if "p_aaer" in BAO.columns:
        print(
            f"distinct enforcement cases behind the positives: "
            f"{int(BAO.loc[BAO['misstate'] == 1, 'p_aaer'].nunique()):,}"
        )
    if "fyear" in BAO.columns:
        rate = BAO.groupby("fyear").agg(
            firm_years=("misstate", "size"), positives=("misstate", "sum")
        )
        rate["share"] = rate["positives"] / rate["firm_years"]
        display(rate)

if BAO_LABELS.is_file():
    labels = pd.read_csv(BAO_LABELS)
    print(f"\nlabel file: {len(labels):,} rows, columns {list(labels.columns)}")
    for column in ("CIK", "P_AAER"):
        if column in labels.columns:
            print(f"distinct {column}: {labels[column].nunique():,}")
    if "YEARA" in labels.columns:
        print(
            f"YEARA (fiscal year of the misstatement) {int(labels['YEARA'].min())} to "
            f"{int(labels['YEARA'].max())}"
        )
        display(labels.groupby("YEARA").size().rename("firm_year_labels").to_frame())
    if "UNDERSTATEMENT" in labels.columns:
        display(labels["UNDERSTATEMENT"].value_counts(dropna=False).rename("rows").to_frame())

## What this audit does not check, and cannot

Stated so that nobody mistakes a clean run for a validated pipeline.

1. **Whether the tag map is right.** The cells above count what each candidate tag resolved.
   They cannot tell you that `Revenues` and
   `RevenueFromContractWithCustomerExcludingAssessedTax` were the same quantity for a given
   filer in a given year. `aaer.clean.xbrl_map.tag_frequency` on one early and one late
   quarter is the check, and it has to be read by a person.
2. **Whether `qtrs == 4` means a four-quarter duration.** Section 5.3 of the Financial
   Statement Data Sets documentation settles it and was not read. Every flow item in the panel
   above depends on that assumption.
3. **The label join.** The listing carries a respondent name and no CIK. Nothing in this
   notebook joins the label side to the financial side, because nothing in this tree can.
4. **Whether the SEC path and the Bao et al. path agree** on the firm-years they share, fiscal
   2009 to 2014. That comparison is the only free check available on the tag mapping's
   accuracy. It is a comparison of two datasets rather than a data audit, so it belongs in the
   next notebook, not this one.
5. **Anything about performance.** No index, no score, no threshold, no metric appears above,
   and none should be added here.